# V7.8 Rebuilt Independent-Data Builder: Deterministic Lock Patch

This is the rebuilt V7.8 workflow with a deterministic lock identity. The lock hashes semantic array contents rather than `.npz`/ZIP container bytes.

Before a lock can be created, two consecutive unchanged runs must produce exactly the same candidate lock ID. If the ID changes, `builder/candidate_lock_diff.csv` records the field(s) responsible.

Keep the same rebuilt V7.8 input structure: `clean/`, `annotated/`, `corrected_masks/`, and `score_manifest.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy


In [ ]:
import cv2,json,hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime,timezone
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
INPUT_ROOT=ROOT/'independent_structural_validation_v7_8_builder';CLEAN_DIR=INPUT_ROOT/'clean';ANNOTATED_DIR=INPUT_ROOT/'annotated';MASK_DIR=INPUT_ROOT/'corrected_masks';MANIFEST_PATH=INPUT_ROOT/'score_manifest.csv'
OUTPUT_ROOT=ROOT/'independent_structural_validation_v7_8_rebuilt';BUILD_DIR=OUTPUT_ROOT/'builder';LOCKS_DIR=OUTPUT_ROOT/'locks';RECON_DIR=OUTPUT_ROOT/'reconstruction_cache';ALIGNED_DIR=OUTPUT_ROOT/'aligned_observed';SEALED_DIR=OUTPUT_ROOT/'sealed'
for d in [BUILD_DIR,LOCKS_DIR,RECON_DIR,ALIGNED_DIR,SEALED_DIR]:d.mkdir(parents=True,exist_ok=True)
CREATE_LOCK=False;EXPECTED_LOCK_ID=None;BUILD_SEALED_OUTCOMES=False;RELEASE_LABELLED_DATASET=False

def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def array_hash(a):
 a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def atomic_json(path,v):
 t=Path(str(path)+'.tmp');t.write_text(json.dumps(v,indent=2,sort_keys=True,default=str));t.replace(path)


## Run the normal rebuilt V7.8 preparation cells first

Run the rebuilt V7.8 cells that create `predictors`, `runtime`, `pages`, `manifest`, `PREDICTOR_PATH` and `INPUT_PATH`. Then run the cell below **instead of the old lock cell**.

This patch is deliberately isolated so it can also be pasted into the existing rebuilt notebook without changing reconstruction semantics.


In [ ]:
required_names=['predictors','runtime','pages','manifest','PREDICTOR_PATH','INPUT_PATH']
missing=[n for n in required_names if n not in globals()]
if missing:raise RuntimeError('Run the rebuilt V7.8 preparation/predictor cells first. Missing: '+', '.join(missing))
semantic_page_hashes={}
for page in sorted(pages):
 z=runtime[page]
 routes=z['routes']
 semantic_page_hashes[page]={
  'aligned_observed':array_hash(z['observed']),
  'removal_mask':array_hash(z['mask']),
  'telea':array_hash(z['telea']),
  'geometry_t80':array_hash(z['recon']),
  **{k:array_hash(routes[k].astype(np.uint8)) for k in ['preserve','remove','ambiguous','horizontal','vertical','compact']},
 }
SETTINGS={'schema':'v7.8-rebuilt-2-deterministic-lock','mask_dilation':2,'minimum_component_area':20,'geometry_threshold':80,'band_width':4,'context_ring_radius':18}
lock_payload={'settings':SETTINGS,'settings_sha256':object_hash(SETTINGS),'predictor_sha256':sha256_file(PREDICTOR_PATH),'input_manifest_sha256':sha256_file(INPUT_PATH),'score_manifest_sha256':sha256_file(MANIFEST_PATH),'components':int(len(predictors)),'pages':int(len(pages)),'source_groups':int(manifest.source_group.nunique()),'semantic_page_hashes':semantic_page_hashes}
LOCK_ID=object_hash(lock_payload);LOCK_PATH=LOCKS_DIR/f'{LOCK_ID}.json';CANDIDATE_PATH=BUILD_DIR/'candidate_lock_payload.json'

def diff_values(a,b,path=''):
 out=[]
 if type(a)!=type(b):return [{'path':path,'previous':a,'current':b}]
 if isinstance(a,dict):
  for k in sorted(set(a)|set(b)):
   p=f'{path}.{k}' if path else str(k)
   if k not in a:out.append({'path':p,'previous':'<missing>','current':b[k]})
   elif k not in b:out.append({'path':p,'previous':a[k],'current':'<missing>'})
   else:out.extend(diff_values(a[k],b[k],p))
 elif isinstance(a,list):
  if a!=b:out.append({'path':path,'previous':a,'current':b})
 elif a!=b:out.append({'path':path,'previous':a,'current':b})
 return out
previous=json.loads(CANDIDATE_PATH.read_text()) if CANDIDATE_PATH.exists() else None
if previous is None:
 print('FIRST DETERMINISTIC CANDIDATE:',LOCK_ID);print('Run the notebook once more unchanged. Do not set CREATE_LOCK=True yet.')
else:
 changes=diff_values(previous['payload'],lock_payload)
 if previous['lock_id']==LOCK_ID:
  print('DETERMINISTIC LOCK CHECK PASSED:',LOCK_ID)
 else:
  pd.DataFrame(changes).to_csv(BUILD_DIR/'candidate_lock_diff.csv',index=False)
  print('CANDIDATE LOCK CHANGED:',previous['lock_id'],'->',LOCK_ID);display(pd.DataFrame(changes).head(100))
CANDIDATE_PATH.write_text(json.dumps({'lock_id':LOCK_ID,'payload':lock_payload},indent=2,sort_keys=True))
if CREATE_LOCK:
 if previous is None or previous.get('lock_id')!=LOCK_ID:raise RuntimeError('LOCK NOT CREATED: run unchanged until DETERMINISTIC LOCK CHECK PASSED.')
 if LOCK_PATH.exists():
  old=json.loads(LOCK_PATH.read_text())
  if old.get('payload')!=lock_payload:raise RuntimeError('Existing lock payload mismatch.')
 else:atomic_json(LOCK_PATH,{'lock_id':LOCK_ID,'payload':lock_payload,'created_utc':datetime.now(timezone.utc).isoformat(),'deterministic_identity':True,'status':'LOCKED_BEFORE_STRUCTURAL_OUTCOMES'})
 print('DATASET LOCK CREATED:',LOCK_ID)
else:print('CREATE_LOCK remains False.')


## Required workflow

1. First run: `CREATE_LOCK=False`. Record the candidate ID.
2. Second unchanged run: it must print `DETERMINISTIC LOCK CHECK PASSED` with exactly the same ID.
3. Only then set `CREATE_LOCK=True` and rerun.
4. If the ID changes, inspect `builder/candidate_lock_diff.csv` rather than creating a lock.

The lock timestamp is metadata written after the lock identity is calculated and therefore does not affect the ID.
